# 02 Traditional Feature Demo

Demonstrates HOG, LBP, and SIFT-BoVW extraction on a small subset.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import hog
from src.preprocess import preprocess_for_features
from src.features import (
    compute_hog_matrix,
    compute_lbp_matrix,
    SIFTBOVWConfig,
    SIFTBOVWExtractor,
)

manifest_path = PROJECT_ROOT / 'outputs' / 'tables' / 'split_manifest.csv'
manifest_df = pd.read_csv(manifest_path)
train_df = manifest_df[manifest_df['split_final'] == 'train_final'].copy()
train_df.head()

In [ ]:
sample_df = train_df.groupby('class_name').head(25).reset_index(drop=True)
sample_paths = sample_df['filepath'].tolist()
sample_labels = sample_df['class_name'].to_numpy()

X_hog = compute_hog_matrix(sample_paths, image_size=(224, 224))
X_lbp = compute_lbp_matrix(sample_paths, image_size=(224, 224))

print('HOG shape:', X_hog.shape)
print('LBP shape:', X_lbp.shape)

In [ ]:
img = preprocess_for_features(sample_paths[0], size=(224, 224), normalize=False)
feat, hog_image = hog(
    img,
    orientations=9,
    pixels_per_cell=(16, 16),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    visualize=True,
    feature_vector=True,
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Input image')
axes[0].axis('off')
axes[1].imshow(hog_image, cmap='gray')
axes[1].set_title('HOG visualization')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
lbp_vec = X_lbp[0]
plt.figure(figsize=(7, 3))
plt.bar(np.arange(len(lbp_vec)), lbp_vec)
plt.title('LBP histogram example')
plt.xlabel('Uniform LBP bin')
plt.ylabel('Normalized frequency')
plt.tight_layout()
plt.show()

In [ ]:
small_paths = sample_paths[:60]
extractor = SIFTBOVWExtractor(SIFTBOVWConfig(vocab_size=64, max_images_for_codebook=60, image_size=(224, 224)))
X_sift = extractor.fit_transform(small_paths)
print('SIFT-BoVW shape:', X_sift.shape)
print('Row L1 norm (first vector):', np.linalg.norm(X_sift[0], ord=1))